In [0]:
from pyspark.sql import functions as F


def validate_events(input_df):
    """Return valid and rejected events without writing any tables."""

    parsed = (
        input_df
        .withColumn(
            "raw_payload",
            F.to_json(F.struct(*[
                F.col(name) for name in input_df.columns
            ]))
        )
        .withColumn("event_id", F.trim(F.col("event_id")))
        .withColumn(
            "order_id",
            F.expr("try_cast(order_id AS BIGINT)")
        )
        .withColumn(
            "customer_id",
            F.expr("try_cast(customer_id AS BIGINT)")
        )
        .withColumn(
            "sequence_number",
            F.expr("try_cast(sequence_number AS BIGINT)")
        )
        .withColumn(
            "amount",
            F.expr("try_cast(amount AS DECIMAL(18,2))")
        )
        .withColumn(
            "event_timestamp",
            F.expr("try_cast(event_time AS TIMESTAMP)")
        )
        .withColumn("event_type", F.upper(F.trim("event_type")))
        .withColumn("status", F.upper(F.trim("status")))
    )

    valid_combination = (
        ((F.col("event_type") == "INSERT") &
         (F.col("status") == "CREATED"))
        |
        ((F.col("event_type") == "UPDATE") &
         F.col("status").isin("PAID", "SHIPPED"))
        |
        ((F.col("event_type") == "CANCEL") &
         (F.col("status") == "CANCELLED"))
    )

    rules = [
        (F.length("event_id") > 0, "Missing event ID"),
        (F.col("order_id") > 0, "Invalid order ID"),
        (F.col("customer_id") > 0, "Invalid customer ID"),
        (F.col("sequence_number") > 0, "Invalid sequence number"),
        (F.col("amount") >= 0, "Invalid or negative amount"),
        (
            F.col("event_timestamp").isNotNull(),
            "Invalid event timestamp"
        ),
        (valid_combination, "Invalid event type/status combination"),
    ]

    checked = parsed.withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            *[
                F.when(
                    ~F.coalesce(condition, F.lit(False)),
                    F.lit(reason)
                )
                for condition, reason in rules
            ]
        )
    )

    return (
        checked.filter(F.col("rejection_reason") == ""),
        checked.filter(F.col("rejection_reason") != "")
    )


# Process the same verified batch.
batch_id = "3d56c37eb8a2483cb213e6020acb2b20"

bronze_df = (
    spark.table("fintech_lakehouse.bronze.orders_raw")
    .filter(F.col("source_file").contains(f"/batch_{batch_id}/"))
)

valid_events, quarantined_events = validate_events(bronze_df)

input_count = bronze_df.count()
valid_count = valid_events.count()
rejected_count = quarantined_events.count()

assert input_count > 0, "Selected batch is empty."
assert input_count == valid_count + rejected_count

print(f"Input events: {input_count}")
print(f"Valid events: {valid_count}")
print(f"Rejected events: {rejected_count}")

In [0]:
from pyspark.sql.window import Window

events_table = "fintech_lakehouse.silver.order_events_clean_dev"
current_table = "fintech_lakehouse.silver.orders_current_dev"
quarantine_table = (
    "fintech_lakehouse.silver.order_events_quarantine_dev"
)

# Compare business content, excluding ingestion metadata.
business_columns = [
    "event_id",
    "order_id",
    "customer_id",
    "event_type",
    "status",
    "amount",
    "sequence_number",
    "event_timestamp",
]

# One event ID must describe one business event.
event_conflicts = (
    valid_events
    .select(*business_columns)
    .distinct()
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

assert event_conflicts.count() == 0, (
    "Conflicting payloads share an event ID. "
    "Inspect event_conflicts before continuing."
)

# Keep one copy of each identical business event.
# The metadata ordering makes the retained source trace predictable.
event_window = Window.partitionBy("event_id").orderBy(
    F.col("ingested_at").asc_nulls_last(),
    F.col("source_file").asc_nulls_last(),
    F.col("raw_payload").asc(),
)

clean_events = (
    valid_events
    .withColumn("_event_rank", F.row_number().over(event_window))
    .filter(F.col("_event_rank") == 1)
    .drop("_event_rank", "rejection_reason")
)

# For this project's contract, an order sequence identifies one event.
sequence_conflicts = (
    clean_events
    .groupBy("order_id", "sequence_number")
    .count()
    .filter(F.col("count") > 1)
)

assert sequence_conflicts.count() == 0, (
    "Multiple events share an order sequence. "
    "Inspect sequence_conflicts before continuing."
)

# Preserve the actual creation timestamp separately from updates.
creation_times = (
    clean_events
    .filter(F.col("event_type") == "INSERT")
    .groupBy("order_id")
    .agg(
        F.min("event_timestamp").alias("order_created_at"),
        F.count("*").alias("_creation_count"),
    )
)

assert creation_times.filter(
    F.col("_creation_count") > 1
).count() == 0, "An order has multiple creation events."

# Build the current-state snapshot from this batch's complete history.
order_window = Window.partitionBy("order_id").orderBy(
    F.col("sequence_number").desc()
)

current_orders = (
    clean_events
    .withColumn("_order_rank", F.row_number().over(order_window))
    .filter(F.col("_order_rank") == 1)
    .drop("_order_rank")
    .join(
        creation_times.drop("_creation_count"),
        on="order_id",
        how="left",
    )
    .withColumn("last_event_at", F.col("event_timestamp"))
    .withColumn("is_cancelled", F.col("status") == "CANCELLED")
    .withColumn("silver_updated_at", F.current_timestamp())
)

# This first exercise requires complete order histories.
assert current_orders.filter(
    F.col("order_created_at").isNull()
).count() == 0, "An order is missing its creation event."

assert current_orders.filter(
    F.col("order_created_at") > F.col("last_event_at")
).count() == 0, "An order has inconsistent timestamps."

clean_count = clean_events.count()
current_count = current_orders.count()

assert current_count == (
    current_orders.select("order_id").distinct().count()
), "Duplicate current order IDs."

# Rebuild only these development tables on each run.
for dataframe, table_name in [
    (clean_events, events_table),
    (current_orders, current_table),
    (quarantined_events, quarantine_table),
]:
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

print(f"Clean events saved: {clean_count}")
print(f"Duplicate copies removed: {valid_count - clean_count}")
print(f"Current orders saved: {current_count}")
print(f"Rejected events saved: {rejected_count}")

display(
    spark.table(current_table)
    .groupBy("status")
    .count()
    .orderBy("status")
)